<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  
  <b>Module 1.2</b>  
  <span style="color:#9ca3af;">RAG vs Fine-Tuning vs Prompt Engineering</span>
</div>


When building an LLM application, you have three primary levers to adapt the model to your specific use case. Understanding when to use which technique is critical for balancing **cost**, **latency**, and **accuracy**.

### 1. Prompt Engineering
Changing the instructions and providing few-shot examples in the prompt.
*   **Pros:** Immediate, no infrastructure cost, easiest to iterate.
*   **Cons:** Limited by context window, context limits latency, doesn't add new permanent knowledge.

### 2. Retrieval-Augmented Generation (RAG)
Fetching external data at inference time and passing it in the prompt.
*   **Pros:** Real-time updates, grounded facts (reduces hallucinations), source attribution, infinite knowledge base.
*   **Cons:** Retrieval overhead (latency), requires vector database infrastructure, complex chunking/retrieval tuning.

### 3. Fine-Tuning
Updating the model's underlying weights using a curated dataset.
*   **Pros:** Teach the model a new format/style, bake in deep domain behaviour, reduces tokens needed in prompt.
*   **Cons:** Expensive and slow to train, knowledge becomes stale immediately, highly susceptible to hallucination without RAG, no source attribution.

## Comparison Matrix

| Dimension | Prompt Engineering | RAG | Fine-Tuning |
| :--- | :--- | :--- | :--- |
| **Cost** | 💲 Very Low | 💲💲 Medium | 💲💲💲 High |
| **Latency** | ⚡ Fast | ⏳ Medium (retrieval overhead) | ⚡ Fast (at inference) |
| **Accuracy on Facts** | Moderate | 🎯 High (grounded) | Moderate (prone to hallucination) |
| **Maintenance** | None | Index refresh | Retrain on new data |
| **Data needed** | None | Unstructured Documents | Labelled QA pairs (high quality) |
| **Source attribution** | ❌ No | ✅ Yes | ❌ No |
| **Best for...** | General tasks, instructions | Dynamic knowledge, fact-checking | Tone, style, new formats |

## Decision Framework Script

Below is an advanced scoring system to help you decide the best approach based on your project requirements.

In [1]:
import pandas as pd

def evaluate_use_case(
    has_external_knowledge: bool,
    knowledge_changes_frequently: bool,
    need_source_attribution: bool,
    need_style_format_change: bool,
    has_labeled_training_data: bool,
    budget_constrained: bool
):
    """
    Evaluates the fit of Prompting, RAG, and Fine-Tuning for a specific use case.
    Returns a Pandas DataFrame with scores and recommendations.
    """
    scores = {
        "Prompt Engineering": 0,
        "RAG": 0,
        "Fine-Tuning": 0
    }
    
    reasons = {
        "Prompt Engineering": [],
        "RAG": [],
        "Fine-Tuning": []
    }

    # Logic for Prompt Engineering
    if budget_constrained:
        scores["Prompt Engineering"] += 2
        reasons["Prompt Engineering"].append("Highly budget-friendly.")
    if not has_external_knowledge and not need_style_format_change:
        scores["Prompt Engineering"] += 3
        reasons["Prompt Engineering"].append("Task relies on general knowledge.")

    # Logic for RAG
    if has_external_knowledge:
        scores["RAG"] += 3
        reasons["RAG"].append("Access to proprietary/external documents.")
    if knowledge_changes_frequently:
        scores["RAG"] += 3
        reasons["RAG"].append("Handles dynamic data easily.")
    if need_source_attribution:
        scores["RAG"] += 4  # Dealbreaker for others
        reasons["RAG"].append("Provides exact source attribution.")

    # Logic for Fine-Tuning
    if need_style_format_change:
        scores["Fine-Tuning"] += 4
        reasons["Fine-Tuning"].append("Best for specific tone/format learning.")
    if has_labeled_training_data:
        scores["Fine-Tuning"] += 2
        reasons["Fine-Tuning"].append("High-quality training data available.")
    if budget_constrained:
        scores["Fine-Tuning"] -= 2
        reasons["Fine-Tuning"].append("Expensive compute/training overhead.")
    if knowledge_changes_frequently:
        scores["Fine-Tuning"] -= 2
        reasons["Fine-Tuning"].append("Hard to keep knowledge up-to-date.")

    # Format output
    results = []
    for approach, score in scores.items():
        results.append({
            "Approach": approach,
            "Fit Score": score,
            "Key Reasons": " | ".join(reasons[approach]) if reasons[approach] else "Not ideal"
        })
    
    df = pd.DataFrame(results).sort_values(by="Fit Score", ascending=False).reset_index(drop=True)
    return df

print("✅ Scoring system loaded.")

✅ Scoring system loaded.


### Test Scenarios
Let's run some common real-world scenarios through the evaluator.

In [2]:
# Scenario 1: Internal Customer Support Bot
print("\n=== SCENARIO 1: Internal Customer Support Bot ===")
print("Goal: Answer questions based on internal Confluence pages. Data changes weekly. Must cite sources.")
df_scen1 = evaluate_use_case(
    has_external_knowledge=True,
    knowledge_changes_frequently=True,
    need_source_attribution=True,
    need_style_format_change=False,
    has_labeled_training_data=False,
    budget_constrained=False
)
display(df_scen1)

# Scenario 2: Medical Report Generator
print("\n=== SCENARIO 2: Medical Report Generator ===")
print("Goal: Take raw doctor notes and output them in a very specific, strict JSON schema and medical jargon.")
df_scen2 = evaluate_use_case(
    has_external_knowledge=False,
    knowledge_changes_frequently=False,
    need_source_attribution=False,
    need_style_format_change=True,
    has_labeled_training_data=True,
    budget_constrained=False
)
display(df_scen2)

# Scenario 3: Indie Developer AI Assistant
print("\n=== SCENARIO 3: Indie Developer AI Assistant ===")
print("Goal: Build a general-purpose coding assistant fast with zero budget.")
df_scen3 = evaluate_use_case(
    has_external_knowledge=False,
    knowledge_changes_frequently=False,
    need_source_attribution=False,
    need_style_format_change=False,
    has_labeled_training_data=False,
    budget_constrained=True
)
display(df_scen3)


=== SCENARIO 1: Internal Customer Support Bot ===
Goal: Answer questions based on internal Confluence pages. Data changes weekly. Must cite sources.


,Approach,Fit Score,Key Reasons
0,RAG,10,Access to proprietary/external documents. | Ha...
1,Prompt Engineering,0,Not ideal
2,Fine-Tuning,-2,Hard to keep knowledge up-to-date.



=== SCENARIO 2: Medical Report Generator ===
Goal: Take raw doctor notes and output them in a very specific, strict JSON schema and medical jargon.


,Approach,Fit Score,Key Reasons
0,Fine-Tuning,6,Best for specific tone/format learning. | High...
1,Prompt Engineering,0,Not ideal
2,RAG,0,Not ideal



=== SCENARIO 3: Indie Developer AI Assistant ===
Goal: Build a general-purpose coding assistant fast with zero budget.


,Approach,Fit Score,Key Reasons
0,Prompt Engineering,5,Highly budget-friendly. | Task relies on gener...
1,RAG,0,Not ideal
2,Fine-Tuning,-2,Expensive compute/training overhead.


## Conclusion

In modern AI engineering, **RAG** has emerged as the default choice for most enterprise applications because it separates the *knowledge base* from the *reasoning engine*.

> 💡 **Pro Tip:** You don't have to choose just one. The most advanced systems use **RAG for knowledge retrieval** and **Fine-Tuning to teach the model how to effectively process and format that retrieved knowledge** (often called RAFT - Retrieval Augmented Fine Tuning).